# Task 2: Unsupervised domain adaptation on PACS
Sources: Photo, Art Painting, Cartoon. Target: Sketch (images only, labels are used in the final evaluation section only)

In [1]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q datasets

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
for p in ["/content/drive/MyDrive/pa1-data/pacs/checkpoints/dann.pt",
          "/content/drive/MyDrive/pa1-data/pacs/checkpoints/cdan.pt",
          "/content/drive/MyDrive/atml-pa1/task2/results/logs/dann.json",
          "/content/drive/MyDrive/atml-pa1/task2/results/logs/cdan.json"]:
    Path(p).unlink(missing_ok=True)

In [3]:
import copy
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from datasets import load_dataset
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torchvision.models import ResNet18_Weights, resnet18

CFG = {
    # protocol shared with task 3
    "seed": 6304,
    "sources": ["photo", "art_painting", "cartoon"],
    "target": "sketch",
    "val_fraction": 0.2,             # 80/20 split inside every source domain
    "resize": 256, "crop": 224,      # random crop + flip for training, center crop otherwise
    "per_source_batch": 8,           # 3 x 8 = 24 source images per update
    "target_batch": 24,              # and 24 unlabeled target images
    "max_epochs": 30, "patience": 5, # stop after 5 epochs without better mean source-val macro-F1
    "lr": 1e-4, "weight_decay": 1e-4,
    # one "source epoch" = one full pass over the largest source training split
    # (the two smaller domains and the target simply keep cycling)

    # DAN
    "lambda_mmd": 1.0,
    "mmd_bandwidth_mults": [0.5, 1.0, 2.0],   # times the median pairwise squared distance in the batch
    # DANN / CDAN
    "disc_hidden": 256, "disc_dropout": 0.5, "domain_loss_weight": 1.0,
    "grl_gamma": 10.0,               # alpha(p) = 2 / (1 + exp(-gamma p)) - 1
    # the manual fixes the optimizer of the network but says nothing about the discriminator's own
    # learning rate. with the same 1e-4 the adversarial game ran away in epoch 1 (domain loss in the
    # thousands, features blowing up); a 10x faster discriminator, as in the CDAN reference code,
    # keeps the game balanced. the discriminator receives the raw 512-d feature as specified.
    "disc_lr_mult": 10.0,
    "disc_input_l2norm": False,      # fallback only: feed the discriminator the L2-normalised feature
    # domain separability probe
    "separability": {"test_size": 0.3, "C": 1.0},
    # controlled study: alignment strength of DAN
    "study_lambdas": [0.1, 1.0, 10.0],
}

# every training run of this notebook. the first four are the main comparison.
RUNS = {
    "source_only": {"method": "source_only"},
    "dan":         {"method": "dan", "lam": 1.0},
    "dann":        {"method": "dann"},
    "cdan":        {"method": "cdan"},
    "dan_lam0.1":  {"method": "dan", "lam": 0.1},
    "dan_lam10":   {"method": "dan", "lam": 10.0},
}
MAIN_RUNS = ["source_only", "dan", "dann", "cdan"]
STUDY_RUNS = ["dan_lam0.1", "dan", "dan_lam10"]          # lambda = 0.1, 1, 10
RUN_LABELS = {"source_only": "Source-only", "dan": "DAN", "dann": "DANN", "cdan": "CDAN",
              "dan_lam0.1": "DAN (lambda=0.1)", "dan_lam10": "DAN (lambda=10)"}

REPO = Path("/content/drive/MyDrive/atml-pa1")
RESULTS = REPO / "task2" / "results"
FIGS = RESULTS / "figures"
LOGS = RESULTS / "logs"
SPLIT_FILE = REPO / "shared" / "splits" / "pacs_sketch_seed6304.json"
CACHE = Path("/content/drive/MyDrive/pa1-data/pacs")     # image tensor + checkpoints, shared with task 3
CKPT = CACHE / "checkpoints"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


In [4]:
if not REPO.exists():
    !git clone https://github.com/mardyweb/atml-pa1.git {REPO}

In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_json(path):
    with open(path) as f:
        return json.load(f)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)


def save_table(df, name):
    """csv for the repo, tex for the report, and show it here."""
    df.to_csv(RESULTS / f"{name}.csv", index=False)
    try:
        df.to_latex(RESULTS / f"{name}.tex", index=False, float_format="%.3f", escape=True)
    except Exception:
        pass
    display(df.round(3))


for folder in [RESULTS, FIGS, LOGS, SPLIT_FILE.parent, CACHE, CKPT]:
    folder.mkdir(parents=True, exist_ok=True)
set_seed(CFG["seed"])
save_json({"config": CFG, "runs": RUNS, "device": str(DEVICE),
           "versions": {"torch": torch.__version__, "torchvision": torchvision.__version__,
                        "sklearn": sklearn.__version__, "numpy": np.__version__}}, RESULTS / "run_info.json")

# ---- same look as the task 1 figures ----
RUN_COLORS = {"source_only": "#2E4057", "dan": "#00798C", "dann": "#D1495B", "cdan": "#EDAE49",
              "dan_lam0.1": "#7FBFC4", "dan_lam10": "#003F4A"}
RUN_MARKERS = {"source_only": "o", "dan": "s", "dann": "D", "cdan": "^", "dan_lam0.1": "v", "dan_lam10": "P"}
INK = "#2B2B2B"
GOOD, BAD = "#00798C", "#D1495B"
TEAL_CMAP = LinearSegmentedColormap.from_list("teal", ["#FBF6EC", "#BFE0DA", "#4FA6A6", "#00798C", "#003F4A"])
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 8.5,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#E4E0D6", "grid.linewidth": 0.7,
    "legend.frameon": False, "legend.fontsize": 8,
    "figure.dpi": 110, "savefig.bbox": "tight", "pdf.fonttype": 42,
})


def save_fig(fig, name):
    fig.savefig(FIGS / f"{name}.pdf")
    fig.savefig(FIGS / f"{name}.png", dpi=300)
    plt.show()
    plt.close(fig)

## Data: PACS, resized once and kept in memory

In [6]:
def build_pacs_tensor():
    """Download PACS and store every image as a 256x256 uint8 tensor."""
    ds = load_dataset("flwrlabs/pacs", split="train")
    class_names = list(ds.features["label"].names)
    size = CFG["resize"]
    images = torch.empty((len(ds), 3, size, size), dtype=torch.uint8)
    domains, labels = [], []
    for i, row in enumerate(ds):
        img = row["image"].convert("RGB").resize((size, size), Image.BILINEAR)
        images[i] = torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1)
        domains.append(row["domain"])
        labels.append(int(row["label"]))
        if (i + 1) % 2000 == 0:
            print(f"  {i + 1}/{len(ds)} images")
    return {"images": images, "domains": domains, "labels": torch.tensor(labels), "class_names": class_names}


# built once (about 2 GB), later sessions and task 3 just load it
if (CACHE / "pacs_256.pt").exists():
    pacs = torch.load(CACHE / "pacs_256.pt")
else:
    pacs = build_pacs_tensor()
    torch.save(pacs, CACHE / "pacs_256.pt")

IMAGES = pacs["images"]                                  # (N, 3, 256, 256) uint8, on the cpu
DOMAIN_OF = np.array(pacs["domains"])
CLASS_NAMES = pacs["class_names"]
N_CLASSES = len(CLASS_NAMES)
SOURCES, TARGET = CFG["sources"], CFG["target"]
TARGET_IDX = np.where(DOMAIN_OF == TARGET)[0]

# labels are split in two on purpose. training code only ever sees SOURCE_LABELS, where every
# target position is -1. the real target labels are used in the final evaluation section only.
_all_labels = pacs["labels"].clone()
SOURCE_LABELS = _all_labels.clone()
SOURCE_LABELS[torch.from_numpy(TARGET_IDX)] = -1
TARGET_LABELS_FINAL_EVAL_ONLY = _all_labels[torch.from_numpy(TARGET_IDX)].numpy()
del _all_labels, pacs

print(CLASS_NAMES)
print({d: int((DOMAIN_OF == d).sum()) for d in SOURCES + [TARGET]})

['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']
{'photo': 1670, 'art_painting': 2048, 'cartoon': 2344, 'sketch': 3929}


In [7]:
# stratified 80/20 split inside every source domain. saved once and reused by task 3.
if SPLIT_FILE.exists():
    SPLITS = load_json(SPLIT_FILE)["splits"]
else:
    SPLITS = {}
    for d in SOURCES:
        idx = np.where(DOMAIN_OF == d)[0]
        tr, va = train_test_split(idx, test_size=CFG["val_fraction"], stratify=SOURCE_LABELS[idx].numpy(),
                                  random_state=CFG["seed"])
        SPLITS[d] = {"train": sorted(tr.tolist()), "val": sorted(va.tolist())}
    save_json({"dataset": "flwrlabs/pacs (hugging face), train split, indices are row numbers",
               "seed": CFG["seed"], "target": TARGET, "splits": SPLITS}, SPLIT_FILE)

STEPS_PER_EPOCH = math.ceil(max(len(SPLITS[d]["train"]) for d in SOURCES) / CFG["per_source_batch"])
print({d: (len(s["train"]), len(s["val"])) for d, s in SPLITS.items()}, "| target:", len(TARGET_IDX),
      "| updates per epoch:", STEPS_PER_EPOCH)

{'photo': (1336, 334), 'art_painting': (1638, 410), 'cartoon': (1875, 469)} | target: 3929 | updates per epoch: 235


## Model and alignment losses

In [8]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


class Net(nn.Module):
    """ImageNet ResNet-18 with a new 7-class linear head. Returns the 512-d feature and the logits."""

    def __init__(self):
        super().__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.fc = nn.Identity()
        self.head = nn.Linear(512, N_CLASSES)

    def forward(self, x_uint8):
        x = (x_uint8.float() / 255 - MEAN) / STD
        feat = self.backbone(x)
        return feat, self.head(feat)


def train_mode(model):
    """Training mode, but BatchNorm keeps its ImageNet running mean/var.
    The BatchNorm scale and shift are ordinary parameters and still get updated."""
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()


class GradReverse(torch.autograd.Function):
    """Identity on the way forward, multiplies the gradient by -alpha on the way back."""

    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None


class Discriminator(nn.Module):
    """source (0) or target (1)? hidden layer -> relu -> dropout -> 2 outputs."""

    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, CFG["disc_hidden"]), nn.ReLU(),
                                 nn.Dropout(CFG["disc_dropout"]), nn.Linear(CFG["disc_hidden"], 2))

    def forward(self, x):
        return self.net(x)


def grl_alpha(progress, alpha_max=1.0):
    """Reversal strength: close to 0 early in training, close to alpha_max at the end."""
    return alpha_max * (2.0 / (1.0 + math.exp(-CFG["grl_gamma"] * progress)) - 1.0)


def mmd2(feat_s, feat_t):
    """Squared MMD between source and target features with a sum of three RBF kernels.
    Bandwidths are 0.5, 1 and 2 times the median pairwise squared distance of the combined batch."""
    z = torch.cat([feat_s, feat_t]).float()
    sq = (z * z).sum(dim=1)
    d2 = (sq[:, None] + sq[None, :] - 2 * z @ z.T).clamp_min(0)          # all pairwise squared distances

    off_diag = ~torch.eye(len(z), dtype=torch.bool, device=z.device)
    median = d2.detach()[off_diag].median().clamp_min(1e-8)              # a constant, no gradient through it
    k = sum(torch.exp(-d2 / (m * median)) for m in CFG["mmd_bandwidth_mults"])

    n = len(feat_s)
    return k[:n, :n].mean() + k[n:, n:].mean() - 2 * k[:n, n:].mean()

## Sampling, augmentation, evaluation

In [9]:
class Cycler:
    """Endless shuffled mini-batches from a fixed set of image indices.
    Reshuffles when it runs out, so a small domain simply starts over."""

    def __init__(self, indices, batch_size, seed):
        self.indices, self.batch_size = np.asarray(indices), batch_size
        self.rng = np.random.default_rng(seed)
        self.queue = []

    def next(self):
        while len(self.queue) < self.batch_size:
            self.queue += self.rng.permutation(self.indices).tolist()
        batch, self.queue = self.queue[:self.batch_size], self.queue[self.batch_size:]
        return batch


def augmented_batch(indices, rng):
    """Random 224 crop + horizontal flip, done on the stored 256x256 tensors."""
    crop, room = CFG["crop"], CFG["resize"] - CFG["crop"]
    out = torch.empty((len(indices), 3, crop, crop), dtype=torch.uint8)
    for k, i in enumerate(indices):
        top, left = rng.integers(0, room + 1, size=2)
        img = IMAGES[i, :, top:top + crop, left:left + crop]
        out[k] = img.flip(-1) if rng.random() < 0.5 else img
    return out.to(DEVICE, non_blocking=True)


@torch.no_grad()
def forward_all(model, indices, batch_size=256):
    """Features and predictions for a list of images (center crop, eval mode)."""
    model.eval()
    off = (CFG["resize"] - CFG["crop"]) // 2
    feats, preds = [], []
    for start in range(0, len(indices), batch_size):
        idx = torch.as_tensor(indices[start:start + batch_size])
        x = IMAGES[idx][:, :, off:off + CFG["crop"], off:off + CFG["crop"]].to(DEVICE)
        f, logits = model(x)
        feats.append(f.float().cpu())
        preds.append(logits.argmax(1).cpu())
    return torch.cat(feats).numpy(), torch.cat(preds).numpy()


def source_validation(model):
    """Accuracy and macro-F1 on each source validation split, plus their means."""
    out = {}
    for d in SOURCES:
        idx = SPLITS[d]["val"]
        _, pred = forward_all(model, idx)
        y = SOURCE_LABELS[idx].numpy()
        out[d] = {"acc": float((pred == y).mean()),
                  "f1": float(f1_score(y, pred, average="macro", labels=list(range(N_CLASSES)), zero_division=0))}
    out["mean_acc"] = float(np.mean([out[d]["acc"] for d in SOURCES]))
    out["mean_f1"] = float(np.mean([out[d]["f1"] for d in SOURCES]))
    return out

## One training loop for every method

In [10]:
def train_run(run_name):
    """Trains one entry of RUNS. Checkpoints are chosen by mean source-validation macro-F1,
    target labels are not available in here."""
    spec = RUNS[run_name]
    method, lam, alpha_max = spec["method"], spec.get("lam", CFG["lambda_mmd"]), spec.get("alpha_max", 1.0)
    ckpt_path, log_path = CKPT / f"{run_name}.pt", LOGS / f"{run_name}.json"
    if ckpt_path.exists() and log_path.exists():
        print(f"{run_name}: already trained, skipping")
        return

    # same seed -> same head init, same source batches and same crops/flips for every method
    set_seed(CFG["seed"])
    model = Net().to(DEVICE)
    disc = None
    if method in ("dann", "cdan"):
        disc = Discriminator(512 if method == "dann" else 512 * N_CLASSES).to(DEVICE)
    groups = [{"params": model.parameters()}]
    if disc is not None:
        groups.append({"params": disc.parameters(), "lr": CFG["lr"] * CFG["disc_lr_mult"]})
    opt = torch.optim.AdamW(groups, lr=CFG["lr"], weight_decay=CFG["weight_decay"])

    # separate random streams, so the target stream can never disturb the source one
    seed = CFG["seed"]
    src_cyclers = [Cycler(SPLITS[d]["train"], CFG["per_source_batch"], [seed, k]) for k, d in enumerate(SOURCES)]
    tgt_cycler = Cycler(TARGET_IDX, CFG["target_batch"], [seed, 99])
    src_aug, tgt_aug = np.random.default_rng([seed, 100]), np.random.default_rng([seed, 101])

    total_steps = CFG["max_epochs"] * STEPS_PER_EPOCH
    n_src = CFG["per_source_batch"] * len(SOURCES)
    domain_y = torch.cat([torch.zeros(n_src), torch.ones(CFG["target_batch"])]).long().to(DEVICE)

    best_f1, best_state, best_epoch, bad_epochs, log, step = -1.0, None, 0, 0, [], 0
    for epoch in range(1, CFG["max_epochs"] + 1):
        t0 = time.time()
        train_mode(model)
        if disc is not None:
            disc.train()
        sums = {"cls": 0.0, "align": 0.0, "dom_acc": 0.0}

        for _ in range(STEPS_PER_EPOCH):
            progress = step / total_steps
            step += 1

            # 8 images from each source domain, labels for the classification loss
            src_idx = sum([c.next() for c in src_cyclers], [])
            ys = SOURCE_LABELS[src_idx].to(DEVICE)
            xs = augmented_batch(src_idx, src_aug)

            if method == "source_only":
                _, logits_s = model(xs)
                cls_loss = F.cross_entropy(logits_s, ys)
                align_loss = torch.zeros((), device=DEVICE)
                loss = cls_loss
            else:
                # 24 unlabeled target images go through the same network
                xt = augmented_batch(tgt_cycler.next(), tgt_aug)
                feat, logits = model(torch.cat([xs, xt]))
                cls_loss = F.cross_entropy(logits[:n_src], ys)        # source images only

                if method == "dan":
                    align_loss = mmd2(feat[:n_src], feat[n_src:])
                    loss = cls_loss + lam * align_loss
                else:
                    # the discriminator gets the raw 512-d feature (fallback switch in CFG), nothing is detached
                    feat_d = F.normalize(feat, dim=1) if CFG["disc_input_l2norm"] else feat
                    if method == "cdan":
                        # outer product of feature and class probabilities, flattened
                        prob = logits.softmax(dim=1)
                        disc_in = torch.bmm(prob.unsqueeze(2), feat_d.unsqueeze(1)).flatten(1)
                    else:
                        disc_in = feat_d
                    alpha = grl_alpha(progress, alpha_max)
                    dom_logits = disc(GradReverse.apply(disc_in, alpha))
                    align_loss = F.cross_entropy(dom_logits, domain_y)   # source and target both count
                    loss = cls_loss + CFG["domain_loss_weight"] * align_loss
                    sums["dom_acc"] += (dom_logits.argmax(1) == domain_y).float().mean().item()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sums["cls"] += cls_loss.item()
            sums["align"] += align_loss.item()

        # model selection on the source validation splits
        val = source_validation(model)
        entry = {"epoch": epoch, "cls_loss": sums["cls"] / STEPS_PER_EPOCH, "align_loss": sums["align"] / STEPS_PER_EPOCH,
                 "disc_acc": sums["dom_acc"] / STEPS_PER_EPOCH if disc is not None else None,
                 "grl_alpha": grl_alpha(step / total_steps, alpha_max) if disc is not None else None, "val": val}
        log.append(entry)
        print(f"{run_name} | epoch {epoch:2d} | cls {entry['cls_loss']:.3f} | align {entry['align_loss']:.4f} | "
              f"val macro-F1 {val['mean_f1']:.4f} | {time.time() - t0:.0f}s")

        if val["mean_f1"] > best_f1:
            best_f1, best_epoch, bad_epochs = val["mean_f1"], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= CFG["patience"]:
                break

    torch.save({"model": best_state, "best_epoch": best_epoch, "best_mean_val_f1": best_f1, "spec": spec}, ckpt_path)
    save_json({"run": run_name, "spec": spec, "best_epoch": best_epoch, "best_mean_val_f1": best_f1,
               "epochs_run": len(log), "steps_per_epoch": STEPS_PER_EPOCH, "log": log}, log_path)
    print(f"{run_name}: kept epoch {best_epoch} (mean source-val macro-F1 {best_f1:.4f})")

## Step 1: Source-only ERM
This checkpoint is also the ERM baseline of task 3.

In [11]:
train_run("source_only")

source_only: already trained, skipping


## Step 2: DAN (MMD alignment)

In [12]:
train_run("dan")

dan: already trained, skipping


## Step 3: DANN (adversarial alignment)

In [13]:
train_run("dann")

dann | epoch  1 | cls 0.491 | align 0.3662 | val macro-F1 0.8930 | 45s
dann | epoch  2 | cls 0.212 | align 0.5891 | val macro-F1 0.9026 | 40s
dann | epoch  3 | cls 0.166 | align 0.6601 | val macro-F1 0.9241 | 41s
dann | epoch  4 | cls 0.123 | align 0.6883 | val macro-F1 0.9109 | 40s
dann | epoch  5 | cls 0.102 | align 0.6896 | val macro-F1 0.9378 | 40s
dann | epoch  6 | cls 0.059 | align 0.6923 | val macro-F1 0.9210 | 40s
dann | epoch  7 | cls 0.086 | align 0.6945 | val macro-F1 0.8928 | 40s
dann | epoch  8 | cls 0.059 | align 0.6929 | val macro-F1 0.9443 | 40s
dann | epoch  9 | cls 0.066 | align 0.6918 | val macro-F1 0.9321 | 40s
dann | epoch 10 | cls 0.072 | align 0.6930 | val macro-F1 0.9230 | 40s
dann | epoch 11 | cls 0.031 | align 0.6932 | val macro-F1 0.9218 | 40s
dann | epoch 12 | cls 0.045 | align 0.6930 | val macro-F1 0.9125 | 40s
dann | epoch 13 | cls 0.047 | align 0.6932 | val macro-F1 0.9399 | 40s
dann: kept epoch 8 (mean source-val macro-F1 0.9443)


## Step 4: CDAN (class-conditional adversarial alignment)

In [14]:
train_run("cdan")

cdan | epoch  1 | cls 0.902 | align 3.8874 | val macro-F1 0.7160 | 41s
cdan | epoch  2 | cls 0.932 | align 1.8193 | val macro-F1 0.8818 | 41s
cdan | epoch  3 | cls 0.234 | align 0.6258 | val macro-F1 0.9027 | 41s
cdan | epoch  4 | cls 0.168 | align 0.6382 | val macro-F1 0.9285 | 41s
cdan | epoch  5 | cls 0.136 | align 0.6688 | val macro-F1 0.9141 | 41s
cdan | epoch  6 | cls 0.090 | align 0.6854 | val macro-F1 0.9377 | 40s
cdan | epoch  7 | cls 0.083 | align 0.6900 | val macro-F1 0.9139 | 40s
cdan | epoch  8 | cls 0.063 | align 0.6922 | val macro-F1 0.9394 | 40s
cdan | epoch  9 | cls 0.059 | align 0.6923 | val macro-F1 0.9356 | 40s
cdan | epoch 10 | cls 0.055 | align 0.6903 | val macro-F1 0.9331 | 40s
cdan | epoch 11 | cls 0.079 | align 0.6960 | val macro-F1 0.9226 | 40s
cdan | epoch 12 | cls 0.083 | align 0.6927 | val macro-F1 0.9312 | 40s
cdan | epoch 13 | cls 0.052 | align 0.6895 | val macro-F1 0.9031 | 40s
cdan: kept epoch 8 (mean source-val macro-F1 0.9394)


## Step 6: Controlled study, DAN with lambda = 0.1 and 10
(lambda = 1 is the DAN run above.) Trained before the final evaluation so that everything is fixed first.

In [15]:
train_run("dan_lam0.1")
train_run("dan_lam10")

dan_lam0.1: already trained, skipping
dan_lam10: already trained, skipping


## Step 5: Final evaluation
Everything above is fixed at this point. This is the only section that reads the target labels.

In [16]:
# write down what was fixed before any target metric exists
missing = [r for r in RUNS if not (CKPT / f"{r}.pt").exists()]
assert not missing, f"train these first: {missing}"
save_json({"locked_at": time.strftime("%Y-%m-%d %H:%M:%S"),
           "selection_rule": "best mean macro-F1 over the three source validation splits",
           "checkpoints": {r: {"best_epoch": load_json(LOGS / f"{r}.json")["best_epoch"],
                               "best_mean_val_f1": load_json(LOGS / f"{r}.json")["best_mean_val_f1"]} for r in RUNS}},
          RESULTS / "locked_before_target_eval.json")

In [ ]:
def load_model(run_name):
    model = Net().to(DEVICE)
    model.load_state_dict(torch.load(CKPT / f"{run_name}.pt", map_location=DEVICE)["model"])
    return model.eval()


def domain_separability(feat_source, feat_target):
    """Can a linear probe tell source from target? 50% = it cannot."""
    sc, seed = CFG["separability"], CFG["seed"]
    # same number of target features as source-validation features
    keep = np.random.default_rng(seed).choice(len(feat_target), size=len(feat_source), replace=False)
    x = np.concatenate([feat_source, feat_target[keep]])
    y = np.concatenate([np.zeros(len(feat_source)), np.ones(len(keep))])
    x_tr, x_te, y_tr, y_te = train_test_split(x, y, test_size=sc["test_size"], stratify=y, random_state=seed)
    probe = LogisticRegression(C=sc["C"], class_weight="balanced", max_iter=5000).fit(x_tr, y_tr)
    return float(probe.score(x_te, y_te))


y_target = TARGET_LABELS_FINAL_EVAL_ONLY
EVAL = {}
for run in RUNS:
    model = load_model(run)
    val = source_validation(model)
    feat_val = np.concatenate([forward_all(model, SPLITS[d]["val"])[0] for d in SOURCES])
    feat_t, pred_t = forward_all(model, TARGET_IDX)
    cm = confusion_matrix(y_target, pred_t, labels=list(range(N_CLASSES)))
    EVAL[run] = {"val": val,
                 "target_acc": float((pred_t == y_target).mean()),
                 "target_f1": float(f1_score(y_target, pred_t, average="macro")),
                 "separability": domain_separability(feat_val, feat_t),
                 "per_class_acc": (cm.diagonal() / cm.sum(1)).tolist(),
                 "confusion": cm.tolist(), "target_pred": pred_t.tolist()}
    del model
save_json(EVAL, RESULTS / "final_evaluation.json")


def result_row(run):
    e, base = EVAL[run], EVAL["source_only"]
    row = {"method": RUN_LABELS[run]}
    for d in SOURCES:
        row[f"{d} acc"], row[f"{d} F1"] = e["val"][d]["acc"], e["val"][d]["f1"]
    row.update({"mean src acc": e["val"]["mean_acc"], "mean src F1": e["val"]["mean_f1"],
                "target acc": e["target_acc"], "target F1": e["target_f1"],
                "target acc change": e["target_acc"] - base["target_acc"], "domain sep.": e["separability"]})
    return row


MAIN_TABLE = pd.DataFrame([result_row(r) for r in MAIN_RUNS])
save_table(MAIN_TABLE, "table_main_comparison")

In [ ]:
# per-class target accuracy and the change against source-only
rows = []
for c, name in enumerate(CLASS_NAMES):
    row = {"class": name, "n_target": int(np.sum(y_target == c))}
    for run in MAIN_RUNS:
        row[RUN_LABELS[run]] = EVAL[run]["per_class_acc"][c]
    for run in MAIN_RUNS[1:]:
        row[f"{RUN_LABELS[run]} change"] = EVAL[run]["per_class_acc"][c] - EVAL["source_only"]["per_class_acc"][c]
    rows.append(row)
PER_CLASS = pd.DataFrame(rows)
save_table(PER_CLASS, "table_per_class_target")

# dominant confusion of every class: which wrong label absorbs it most often
rows = []
for run in MAIN_RUNS:
    cm = np.array(EVAL[run]["confusion"])
    for c, name in enumerate(CLASS_NAMES):
        wrong = cm[c].copy()
        wrong[c] = 0
        rows.append({"method": RUN_LABELS[run], "true class": name, "accuracy": cm[c, c] / cm[c].sum(),
                     "most confused with": CLASS_NAMES[int(wrong.argmax())] if wrong.max() > 0 else "nothing",
                     "share of class": wrong.max() / cm[c].sum()})
CONFUSIONS = pd.DataFrame(rows)
CONFUSIONS.to_csv(RESULTS / "table_dominant_confusions.csv", index=False)

# biggest winner and loser class of each adaptation method, with its main confusion before and after
for run in MAIN_RUNS[1:]:
    change = PER_CLASS[f"{RUN_LABELS[run]} change"].values
    for tag, c in [("largest gain", int(change.argmax())), ("largest drop", int(change.argmin()))]:
        before = CONFUSIONS[(CONFUSIONS.method == "Source-only") & (CONFUSIONS["true class"] == CLASS_NAMES[c])].iloc[0]
        after = CONFUSIONS[(CONFUSIONS.method == RUN_LABELS[run]) & (CONFUSIONS["true class"] == CLASS_NAMES[c])].iloc[0]
        print(f"{RUN_LABELS[run]:5s} {tag}: {CLASS_NAMES[c]:9s} {100 * change[c]:+5.1f} pp | "
              f"source-only confuses it with {before['most confused with']} ({100 * before['share of class']:.0f}%), "
              f"{RUN_LABELS[run]} with {after['most confused with']} ({100 * after['share of class']:.0f}%)")

In [ ]:
# controlled study table: alignment strength of DAN (source-only = no alignment, for reference)
rows = []
for run, lam in [("source_only", 0.0)] + list(zip(STUDY_RUNS, CFG["study_lambdas"])):
    e = EVAL[run]
    rows.append({"lambda": lam, "mean src acc": e["val"]["mean_acc"], "mean src F1": e["val"]["mean_f1"],
                 "domain sep.": e["separability"], "target acc": e["target_acc"], "target F1": e["target_f1"],
                 "best epoch": load_json(LOGS / f"{run}.json")["best_epoch"]})
STUDY_TABLE = pd.DataFrame(rows)
save_table(STUDY_TABLE, "table_alignment_strength_study")

## Figures

In [ ]:
# training curves: did every method train the way it should?
logs = {r: load_json(LOGS / f"{r}.json") for r in RUNS}
fig, axes = plt.subplots(1, 4, figsize=(7.0, 2.05))


def curve(ax, run, key, **kw):
    ep = [e["epoch"] for e in logs[run]["log"]]
    vals = [e[key] if key != "val_f1" else e["val"]["mean_f1"] for e in logs[run]["log"]]
    ax.plot(ep, vals, color=RUN_COLORS[run], marker=RUN_MARKERS[run], markersize=3, linewidth=1.4,
            label=RUN_LABELS[run], **kw)


for run in MAIN_RUNS:
    curve(axes[0], run, "cls_loss")
    curve(axes[3], run, "val_f1")
for run in STUDY_RUNS:
    curve(axes[1], run, "align_loss")
for run in ["dann", "cdan"]:
    curve(axes[2], run, "align_loss")
    curve(axes[2], run, "disc_acc", linestyle=":")
axes[2].axhline(math.log(2), color="#999999", linewidth=0.8, linestyle="--")      # loss of a coin-flip discriminator
for ax, title in zip(axes, ["source class. loss", "DAN: MMD$^2$", "domain loss (solid)\ndisc. accuracy (dotted)",
                            "mean source-val macro-F1"]):
    ax.set_title(title, fontsize=8)
    ax.set_xlabel("epoch")
handles = [Line2D([], [], color=RUN_COLORS[r], marker=RUN_MARKERS[r], markersize=4, label=RUN_LABELS[r]) for r in RUNS]
fig.legend(handles=handles, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.12), fontsize=7, columnspacing=1.0)
fig.tight_layout(w_pad=0.6)
save_fig(fig, "fig_training_curves")

In [ ]:
# per-class change on the target against source-only
fig, ax = plt.subplots(figsize=(7.0, 2.4))
width = 0.26
for j, run in enumerate(MAIN_RUNS[1:]):
    vals = 100 * PER_CLASS[f"{RUN_LABELS[run]} change"].values
    bars = ax.bar(np.arange(N_CLASSES) + (j - 1) * width, vals, width, color=RUN_COLORS[run], edgecolor="white",
                  linewidth=0.6, label=RUN_LABELS[run])
    for b, v in zip(bars, vals):
        ax.annotate(f"{v:+.0f}", xy=(b.get_x() + b.get_width() / 2, v), xytext=(0, 2 if v >= 0 else -2),
                    textcoords="offset points", ha="center", va="bottom" if v >= 0 else "top", fontsize=6, color=INK)
ax.axhline(0, color=INK, linewidth=0.8)
ax.set_xticks(np.arange(N_CLASSES))
ax.set_xticklabels([f"{n}\n{100 * a:.0f}%" for n, a in zip(CLASS_NAMES, PER_CLASS["Source-only"])], fontsize=8)
ax.set_xlabel("target class (source-only accuracy underneath)")
ax.set_ylabel("accuracy change vs source-only (pp)")
ax.grid(axis="x", visible=False)
ax.margins(y=0.18)
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.2))
save_fig(fig, "fig_per_class_target_change")

In [ ]:
# confusion matrices on the target, rows add up to 100%
fig, axes = plt.subplots(1, 4, figsize=(7.0, 2.25))
short = [n[:4] for n in CLASS_NAMES]
for ax, run in zip(axes, MAIN_RUNS):
    cm = np.array(EVAL[run]["confusion"], dtype=float)
    cm = 100 * cm / cm.sum(1, keepdims=True)
    ax.imshow(cm, cmap=TEAL_CMAP, vmin=0, vmax=100)
    for r in range(N_CLASSES):
        for c in range(N_CLASSES):
            if cm[r, c] >= 10:          # small entries stay blank to keep it readable
                ax.text(c, r, f"{cm[r, c]:.0f}", ha="center", va="center", fontsize=5.5,
                        color="white" if cm[r, c] > 55 else INK)
    ax.set_title(f"{RUN_LABELS[run]} ({100 * EVAL[run]['target_acc']:.1f}%)", fontsize=8, color=RUN_COLORS[run],
                 fontweight="bold")
    ax.set_xticks(range(N_CLASSES))
    ax.set_xticklabels(short, rotation=90, fontsize=6.5)
    ax.set_yticks(range(N_CLASSES))
    ax.set_yticklabels(short if ax is axes[0] else [], fontsize=6.5)
    ax.grid(False)
axes[0].set_ylabel("true class")
fig.supxlabel("predicted class", fontsize=8, y=-0.04)
fig.tight_layout(w_pad=0.4)
save_fig(fig, "fig_target_confusions")

In [ ]:
# does a less separable representation mean better target accuracy?
fig, ax = plt.subplots(figsize=(3.4, 2.7))
for run in RUNS:
    ax.scatter(100 * EVAL[run]["separability"], 100 * EVAL[run]["target_acc"], s=60, color=RUN_COLORS[run],
               marker=RUN_MARKERS[run], edgecolor="white", linewidth=0.7, zorder=3, label=RUN_LABELS[run])
ax.axvline(50, color="#999999", linewidth=0.8, linestyle="--")
ax.set_xlabel("domain separability (%), 50 = chance")
ax.set_ylabel("target accuracy (%)")
fig.legend(loc="center left", bbox_to_anchor=(0.97, 0.5), fontsize=7)
save_fig(fig, "fig_separability_vs_target")

In [ ]:
# controlled study: what stronger MMD pressure does
fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.0))
lams = CFG["study_lambdas"]
for ax, col, title in zip(axes, ["mean src F1", "domain sep.", "target acc"],
                          ["mean source-val macro-F1 (%)", "domain separability (%)", "target accuracy (%)"]):
    study = STUDY_TABLE[STUDY_TABLE["lambda"] > 0]
    ax.plot(lams, 100 * study[col].values, color=RUN_COLORS["dan"], marker="s", markersize=5, linewidth=1.8,
            markeredgecolor="white", label="DAN")
    ax.axhline(100 * STUDY_TABLE[col].values[0], color=RUN_COLORS["source_only"], linestyle="--", linewidth=1.2,
               label="Source-only")
    ax.set_xscale("log")
    ax.set_xticks(lams)
    ax.set_xticklabels([str(l) for l in lams])
    ax.minorticks_off()
    ax.set_xlabel("MMD weight lambda")
    ax.set_title(title, fontsize=8)
axes[0].legend(fontsize=7)
fig.tight_layout(w_pad=0.8)
save_fig(fig, "fig_alignment_strength_study")

In [ ]:
# a few target failures: sketches of the class source-only handles worst, with every method's answer
worst = int(np.argmin(EVAL["source_only"]["per_class_acc"]))
pred_base = np.array(EVAL["source_only"]["target_pred"])
wrong = np.where((y_target == worst) & (pred_base != worst))[0]
show = np.random.default_rng(CFG["seed"]).permutation(wrong)[:5]
short_name = {**RUN_LABELS, "source_only": "Src-only"}

fig, axes = plt.subplots(1, 5, figsize=(7.0, 2.4))
for ax in axes:
    ax.axis("off")
for ax, j in zip(axes, show):
    ax.imshow(IMAGES[TARGET_IDX[j]].permute(1, 2, 0).numpy())
    ax.set_title(f"true: {CLASS_NAMES[worst]}", fontsize=7.5)
    for k, run in enumerate(MAIN_RUNS):
        p = EVAL[run]["target_pred"][j]
        ax.text(0.0, -0.06 - 0.125 * k, f"{short_name[run]}: {CLASS_NAMES[p]}", transform=ax.transAxes, fontsize=6.5,
                fontweight="bold", va="top", ha="left", color=GOOD if p == worst else BAD)
fig.subplots_adjust(bottom=0.4, wspace=0.08)
save_fig(fig, "fig_target_failures")

## Commit to GitHub

In [ ]:
from google.colab import userdata
token = userdata.get("GH_TOKEN")
%cd {REPO}
!git config user.name "mardyweb"
!git config user.email "maryamw17@outlook.com"
!git remote set-url origin https://{token}@github.com/mardyweb/atml-pa1.git
!git add task2 shared
!git commit -m "Task 2 (rerun): notebook, splits, results and figures"
!git push